# Data Loading, Cleaning & Export

### Importazione Librerie

In [32]:
import numpy as np
import pandas as pd
import requests

### Caricamento Dati

In [159]:
CSV_files = {
    "order_items":         r"output/orders_items.csv",
    "order_payments":      r"dataset/olist_order_payments_dataset.csv",
    "order_dataset":       r"output/order_dataset.csv",
    "list_product":        r"dataset/olist_products_dataset.csv",
    "og_product_list":     r"dataset/olist_products_dataset.csv"
}
dataframes = {name: pd.read_csv(path) for name, path in CSV_files.items()}

df_orders_items        = dataframes["order_items"]
df_order_payments      = dataframes["order_payments"]
df_order_dataset       = dataframes["order_dataset"]
df_list_product        = dataframes["list_product"]
df_og_product_list     = dataframes["og_product_list"]

### Regressione Lineare Multivariata

In [160]:
order_it=df_orders_items.drop(['eur_price','eur_freight_value','shipping_limit_date','seller_id'],axis=1)

In [161]:
product=df_list_product.drop(['product_name_lenght','product_description_lenght','product_photos_qty','product_category_name'],axis=1)
df_list_product

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [162]:
order_product=order_it.merge(product, on='product_id')
order_product=order_product.dropna()
order_product.info()

<class 'pandas.DataFrame'>
Index: 112632 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   order_id           112632 non-null  str    
 1   order_item_id      112632 non-null  int64  
 2   product_id         112632 non-null  str    
 3   product_weight_g   112632 non-null  float64
 4   product_length_cm  112632 non-null  float64
 5   product_height_cm  112632 non-null  float64
 6   product_width_cm   112632 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 6.9 MB


In [163]:
order_product=order_product.drop(['product_id','order_item_id'],axis=1)

In [164]:
order_pp=order_product.merge(df_order_payments, on='order_id')
order_pp

,order_id,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_type,payment_installments,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,650.0,28.0,9.0,14.0,1,credit_card,2,72.19
1,00018f77f2f0320c557190d7a144bdd3,30000.0,50.0,30.0,40.0,1,credit_card,3,259.83
2,000229ec398224ef6ca0657da4fc703e,3050.0,33.0,13.0,33.0,1,credit_card,5,216.87
3,00024acbcdf0a6daa1e931b038114c75,200.0,16.0,10.0,15.0,1,credit_card,2,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,3750.0,35.0,40.0,30.0,1,credit_card,3,218.04
...,...,...,...,...,...,...,...,...,...
117576,fffc94f6ce00a00581880bf54a75a037,10150.0,89.0,15.0,40.0,1,boleto,1,343.40
117577,fffcd46ef2263f404302a634eb57f7eb,8950.0,45.0,26.0,38.0,1,boleto,1,386.53
117578,fffce4705a9662cd70adb13d4a31832d,967.0,21.0,24.0,19.0,1,credit_card,3,116.85
117579,fffe18544ffabc95dfada21779c9644f,100.0,20.0,20.0,20.0,1,credit_card,3,64.71


In [165]:
df_order_dataset=df_order_dataset[df_order_dataset['order_status']=='delivered']#filtarer solo i consegnati


In [166]:
df_order_dataset=df_order_dataset.drop(['order_status','customer_id'],axis=1)

In [167]:
df=order_pp.merge(df_order_dataset, on='order_id')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115015 entries, 0 to 115014
Data columns (total 16 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       115015 non-null  str    
 1   product_weight_g               115015 non-null  float64
 2   product_length_cm              115015 non-null  float64
 3   product_height_cm              115015 non-null  float64
 4   product_width_cm               115015 non-null  float64
 5   payment_sequential             115015 non-null  int64  
 6   payment_type                   115015 non-null  str    
 7   payment_installments           115015 non-null  int64  
 8   payment_value                  115015 non-null  float64
 9   order_purchase_timestamp       115015 non-null  str    
 10  order_approved_at              115000 non-null  str    
 11  order_delivered_carrier_date   115013 non-null  str    
 12  order_delivered_customer_date  115007 non

In [168]:

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')  # 'coerce' trasforma valori non validi in NaN

In [169]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115015 entries, 0 to 115014
Data columns (total 16 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       115015 non-null  str           
 1   product_weight_g               115015 non-null  float64       
 2   product_length_cm              115015 non-null  float64       
 3   product_height_cm              115015 non-null  float64       
 4   product_width_cm               115015 non-null  float64       
 5   payment_sequential             115015 non-null  int64         
 6   payment_type                   115015 non-null  str           
 7   payment_installments           115015 non-null  int64         
 8   payment_value                  115015 non-null  float64       
 9   order_purchase_timestamp       115015 non-null  datetime64[us]
 10  order_approved_at              115000 non-null  datetime64[us]
 11  order_deliv

In [170]:
df['actual_delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

In [171]:
# tempo di approvazione
df['approval_time_days'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.days

# tempo spedizione dal corriere al cliente
df['shipping_time_days'] = (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days

# mese e giorno settimana dell'acquisto
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_weekday'] = df['order_purchase_timestamp'].dt.weekday

In [172]:
df = pd.get_dummies(df, columns=['payment_type'], drop_first=True, dtype=int)

In [173]:
df = df.dropna(subset=['actual_delivery_days', 'approval_time_days', 'shipping_time_days'])

In [174]:
df.info()

<class 'pandas.DataFrame'>
Index: 114991 entries, 0 to 115014
Data columns (total 22 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       114991 non-null  str           
 1   product_weight_g               114991 non-null  float64       
 2   product_length_cm              114991 non-null  float64       
 3   product_height_cm              114991 non-null  float64       
 4   product_width_cm               114991 non-null  float64       
 5   payment_sequential             114991 non-null  int64         
 6   payment_installments           114991 non-null  int64         
 7   payment_value                  114991 non-null  float64       
 8   order_purchase_timestamp       114991 non-null  datetime64[us]
 9   order_approved_at              114991 non-null  datetime64[us]
 10  order_delivered_carrier_date   114991 non-null  datetime64[us]
 11  order_delivered_

In [175]:
features = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'payment_sequential',
    'payment_installments'
] + [col for col in df.columns if col.startswith('payment_type_')]

In [176]:
y = df['actual_delivery_days']
X = df[features]

In [177]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print("R²:", model.score(X_test, y_test))

R²: 0.015378555995066212
